In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sqlalchemy import create_engine, text

In [2]:
ucdd_counties = ['Cannon', 'Clay', 'Cumberland', 'DeKalb', 'Fentress', 'Jackson',
                  'Macon', 'Overton', 'Pickett', 'Putnam', 'Smith', 'Van Buren',
                  'Warren', 'White']

***EXPLORATION***

In [3]:
thda_raw = pd.read_excel('thda_housing_needs_ucdd.csv', sheet_name='Data 2026')
thda_raw.shape

(95, 220)

In [4]:
thda_raw.columns

Index(['Geography', 'p_white_25000_less', 'p_white_25000_49999',
       'p_white_50000_74999', 'p_white_75000_99999', 'p_white_100000_plus',
       'p_black_25000_less', 'p_black_25000_49999', 'p_black_50000_74999',
       'p_black_75000_99999',
       ...
       'White_DR', 'NHOPI_DR', 'ALL_DR', 'FHEO_color', 'FHEO_disability',
       'FHEO_fam_stat', 'FHEO_nat_org', 'FHEO_race', 'FHEO_retal', 'FHEO_sex'],
      dtype='object', length=220)

In [5]:
thda_raw.head()

,Geography,p_white_25000_less,p_white_25000_49999,p_white_50000_74999,p_white_75000_99999,p_white_100000_plus,p_black_25000_less,p_black_25000_49999,p_black_50000_74999,p_black_75000_99999,...,White_DR,NHOPI_DR,ALL_DR,FHEO_color,FHEO_disability,FHEO_fam_stat,FHEO_nat_org,FHEO_race,FHEO_retal,FHEO_sex
0,"Anderson County, Tennessee",17.854451,20.228877,16.278353,12.320976,33.317344,25.836910,27.896996,9.785408,9.957082,...,6.364922,50.0,6.812933,1,11,1,0,3,0,1
1,"Bedford County, Tennessee",10.827861,19.434788,19.660100,17.477791,32.599459,12.034574,49.401596,16.688830,10.571809,...,8.043478,0.0,8.595041,0,2,0,0,0,0,0
2,"Benton County, Tennessee",21.039793,26.940082,19.728617,13.218478,19.073029,13.812155,56.353591,29.834254,0.000000,...,12.000000,0.0,13.043478,0,0,0,0,0,0,0
3,"Bledsoe County, Tennessee",24.060803,20.608035,22.410423,10.922910,21.997828,40.000000,12.000000,0.000000,0.000000,...,10.000000,0.0,10.606061,0,0,0,0,0,0,0
4,"Blount County, Tennessee",11.418406,17.195424,18.852459,15.681881,36.851830,18.597786,19.409594,19.261993,12.029520,...,7.235890,0.0,7.629256,1,4,3,0,7,0,0


In [6]:
thda_raw.isnull().sum()

Geography              0
p_white_25000_less     0
p_white_25000_49999    0
p_white_50000_74999    0
p_white_75000_99999    0
                      ..
FHEO_fam_stat          0
FHEO_nat_org           0
FHEO_race              0
FHEO_retal             0
FHEO_sex               0
Length: 220, dtype: int64

In [7]:
picture_raw = pd.read_csv('hud_picture_subsidized_county_tn.csv')
picture_raw.shape

(13, 84)

In [8]:
picture_raw.columns

Index(['Summary level', 'Program label', 'Program', 'Sub-program', 'Name',
       'Code', 'Subsidized units available', '# Occupied Units', '% Occupied',
       '# Reported', '% Reported', 'Average months since report',
       '% moved in past year', 'Number of people per unit',
       'Number of people: total', 'Average Family Expenditure per month ($$)',
       'Average HUD Expenditure per month ($$)', 'Household income per year',
       'Household income per year per person', '% No Income', '% $1 - $4,999',
       '% $5,000 - $9,999', '% $10,000 - $14,999', '% $15,000 - $19,999',
       '% $20,000 - $24,999', '% $25,000 - $29,999', '% $30,000 - $39,999',
       '% $40,000 or more',
       '% Households where wages are major source of income',
       '% Households where welfare is major source of income',
       '% Households with other major sources of income',
       '% of local median (Household income)', '% low income',
       '% very low income', '% extremely low income',
      

In [9]:
picture_raw.head()

,Summary level,Program label,Program,Sub-program,Name,Code,Subsidized units available,# Occupied Units,% Occupied,# Reported,...,% minority (Census tract),% single family owners (Census tract),Congressional District,CBSA,PLACE,Latitude,Longitude,State,PHA Total Units,HA category
0,9,202/PRAC,8,NaN,Cannon County,47015,18,17,97,17,...,12,79,NaN,NaN,NaN,NaN,NaN,TN,NaN,NaN
1,9,202/PRAC,8,NaN,Cumberland County,47035,30,29,95,28,...,12,64,NaN,NaN,NaN,NaN,NaN,TN,NaN,NaN
2,9,811/PRAC,9,NaN,Cumberland County,47035,17,12,70,13,...,8,92,NaN,NaN,NaN,NaN,NaN,TN,NaN,NaN
3,9,202/PRAC,8,NaN,DeKalb County,47041,26,25,95,25,...,17,73,NaN,NaN,NaN,NaN,NaN,TN,NaN,NaN
4,9,811/PRAC,9,NaN,DeKalb County,47041,6,-4,-4,-4,...,-4,-4,NaN,NaN,NaN,NaN,NaN,TN,NaN,NaN


In [10]:
picture_raw.isnull().sum()

Summary level       0
Program label       0
Program             0
Sub-program        13
Name                0
                   ..
Latitude           13
Longitude          13
State               0
PHA Total Units    13
HA category        13
Length: 84, dtype: int64

In [11]:
gis_raw = pd.read_csv('hud_gis_multifamily_properties_tn.csv',
                       usecols=['STD_ST', 'CURCNTY_NM', 'IS_202_811_IND', 'TOTAL_ASSISTED_UNIT_COUNT',
                                'TOTAL_UNIT_COUNT', 'PROPERTY_NAME_TEXT', 'LAT', 'LON'])
gis_raw.shape

(23781, 8)

In [12]:
gis_raw.dtypes

PROPERTY_NAME_TEXT            object
TOTAL_ASSISTED_UNIT_COUNT      int64
TOTAL_UNIT_COUNT               int64
IS_202_811_IND                object
CURCNTY_NM                    object
STD_ST                        object
LAT                          float64
LON                          float64
dtype: object

In [13]:
gis_raw['STD_ST'].value_counts()

STD_ST
CA    1795
OH    1393
NY    1362
PA    1053
TX    1010
IL     944
NC     911
MA     902
MI     712
MN     700
NJ     693
FL     689
WI     655
GA     623
TN     608
MD     511
IN     495
MO     490
KY     488
VA     474
WA     435
CT     410
SC     406
AL     401
MS     346
LA     344
CO     343
AR     342
OR     329
ME     296
KS     274
OK     263
IA     262
RI     248
NH     239
WV     218
NE     214
AZ     200
PR     196
SD     190
VT     148
NM     136
ID     131
DE     128
MT     117
UT     106
HI     105
ND     101
DC      99
AK      75
NV      65
WY      65
VI      20
MP       4
GU       1
Name: count, dtype: int64

In [14]:
age_sex_raw = pd.read_csv('age_and_sex_by_county.csv')
age_sex_raw.shape

(42, 181)

In [15]:
age_sex_raw.columns

Index(['Label (Grouping)', 'Tennessee!!Total!!Estimate',
       'Tennessee!!Total!!Margin of Error', 'Tennessee!!Percent!!Estimate',
       'Tennessee!!Percent!!Margin of Error', 'Tennessee!!Male!!Estimate',
       'Tennessee!!Male!!Margin of Error', 'Tennessee!!Percent Male!!Estimate',
       'Tennessee!!Percent Male!!Margin of Error',
       'Tennessee!!Female!!Estimate',
       ...
       'White County, Tennessee!!Percent!!Estimate',
       'White County, Tennessee!!Percent!!Margin of Error',
       'White County, Tennessee!!Male!!Estimate',
       'White County, Tennessee!!Male!!Margin of Error',
       'White County, Tennessee!!Percent Male!!Estimate',
       'White County, Tennessee!!Percent Male!!Margin of Error',
       'White County, Tennessee!!Female!!Estimate',
       'White County, Tennessee!!Female!!Margin of Error',
       'White County, Tennessee!!Percent Female!!Estimate',
       'White County, Tennessee!!Percent Female!!Margin of Error'],
      dtype='object', length=1

In [16]:
age_sex_raw.head()

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!Percent!!Estimate,Tennessee!!Percent!!Margin of Error,Tennessee!!Male!!Estimate,Tennessee!!Male!!Margin of Error,Tennessee!!Percent Male!!Estimate,Tennessee!!Percent Male!!Margin of Error,Tennessee!!Female!!Estimate,...,"White County, Tennessee!!Percent!!Estimate","White County, Tennessee!!Percent!!Margin of Error","White County, Tennessee!!Male!!Estimate","White County, Tennessee!!Male!!Margin of Error","White County, Tennessee!!Percent Male!!Estimate","White County, Tennessee!!Percent Male!!Margin of Error","White County, Tennessee!!Female!!Estimate","White County, Tennessee!!Female!!Margin of Error","White County, Tennessee!!Percent Female!!Estimate","White County, Tennessee!!Percent Female!!Margin of Error"
0,Total population,"7,066,383",*****,(X),(X),"3,467,734","±1,520",(X),(X),"3,598,649",...,(X),(X),"13,952",±186,(X),(X),"14,208",±186,(X),(X)
1,AGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Under 5 years,"410,924",±873,5.8%,±0.1,"210,517",±944,6.1%,±0.1,"200,407",...,5.5%,±0.4,826,±99,5.9%,±0.7,710,±78,5.0%,±0.5
3,5 to 9 years,"427,842","±4,605",6.1%,±0.1,"219,155","±3,363",6.3%,±0.1,"208,687",...,6.3%,±0.8,911,±166,6.5%,±1.2,851,±123,6.0%,±0.9
4,10 to 14 years,"449,752","±4,716",6.4%,±0.1,"230,909","±3,513",6.7%,±0.1,"218,843",...,6.0%,±0.8,860,±193,6.2%,±1.4,818,±149,5.8%,±1.0


In [17]:
disability_raw = pd.read_csv('disability_characteristics_by_county.csv')
disability_raw.shape

(73, 91)

In [18]:
disability_raw.columns

Index(['Label (Grouping)', 'Tennessee!!Total!!Estimate',
       'Tennessee!!Total!!Margin of Error',
       'Tennessee!!With a disability!!Estimate',
       'Tennessee!!With a disability!!Margin of Error',
       'Tennessee!!Percent with a disability!!Estimate',
       'Tennessee!!Percent with a disability!!Margin of Error',
       'Cannon County, Tennessee!!Total!!Estimate',
       'Cannon County, Tennessee!!Total!!Margin of Error',
       'Cannon County, Tennessee!!With a disability!!Estimate',
       'Cannon County, Tennessee!!With a disability!!Margin of Error',
       'Cannon County, Tennessee!!Percent with a disability!!Estimate',
       'Cannon County, Tennessee!!Percent with a disability!!Margin of Error',
       'Clay County, Tennessee!!Total!!Estimate',
       'Clay County, Tennessee!!Total!!Margin of Error',
       'Clay County, Tennessee!!With a disability!!Estimate',
       'Clay County, Tennessee!!With a disability!!Margin of Error',
       'Clay County, Tennessee!!Percen

In [19]:
disability_raw.head()

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!With a disability!!Estimate,Tennessee!!With a disability!!Margin of Error,Tennessee!!Percent with a disability!!Estimate,Tennessee!!Percent with a disability!!Margin of Error,"Cannon County, Tennessee!!Total!!Estimate","Cannon County, Tennessee!!Total!!Margin of Error","Cannon County, Tennessee!!With a disability!!Estimate",...,"Warren County, Tennessee!!With a disability!!Estimate","Warren County, Tennessee!!With a disability!!Margin of Error","Warren County, Tennessee!!Percent with a disability!!Estimate","Warren County, Tennessee!!Percent with a disability!!Margin of Error","White County, Tennessee!!Total!!Estimate","White County, Tennessee!!Total!!Margin of Error","White County, Tennessee!!With a disability!!Estimate","White County, Tennessee!!With a disability!!Margin of Error","White County, Tennessee!!Percent with a disability!!Estimate","White County, Tennessee!!Percent with a disability!!Margin of Error"
0,Total civilian noninstitutionalized population,"6,970,655","±1,325","1,041,249","±8,879",14.9%,±0.1,"14,692",±19,"2,751",...,"7,524",±601,18.0%,±1.4,"27,842",±54,"5,066",±557,18.2%,±2.0
1,SEX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Male,"3,400,177","±2,098","502,972","±4,860",14.8%,±0.1,"7,501",±118,"1,678",...,"3,706",±422,17.8%,±2.0,"13,762",±205,"2,496",±383,18.1%,±2.8
3,Female,"3,570,478","±1,779","538,277","±5,867",15.1%,±0.2,"7,191",±116,"1,073",...,"3,818",±424,18.3%,±2.0,"14,080",±193,"2,570",±356,18.3%,±2.4
4,RACE AND HISPANIC OR LATINO ORIGIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
poverty_raw = pd.read_csv('poverty_status_by_county.csv')
poverty_raw.shape

(69, 91)

In [21]:
poverty_raw.columns

Index(['Label (Grouping)', 'Tennessee!!Total!!Estimate',
       'Tennessee!!Total!!Margin of Error',
       'Tennessee!!Below poverty level!!Estimate',
       'Tennessee!!Below poverty level!!Margin of Error',
       'Tennessee!!Percent below poverty level!!Estimate',
       'Tennessee!!Percent below poverty level!!Margin of Error',
       'Cannon County, Tennessee!!Total!!Estimate',
       'Cannon County, Tennessee!!Total!!Margin of Error',
       'Cannon County, Tennessee!!Below poverty level!!Estimate',
       'Cannon County, Tennessee!!Below poverty level!!Margin of Error',
       'Cannon County, Tennessee!!Percent below poverty level!!Estimate',
       'Cannon County, Tennessee!!Percent below poverty level!!Margin of Error',
       'Clay County, Tennessee!!Total!!Estimate',
       'Clay County, Tennessee!!Total!!Margin of Error',
       'Clay County, Tennessee!!Below poverty level!!Estimate',
       'Clay County, Tennessee!!Below poverty level!!Margin of Error',
       'Clay Count

In [22]:
poverty_raw.head()

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!Below poverty level!!Estimate,Tennessee!!Below poverty level!!Margin of Error,Tennessee!!Percent below poverty level!!Estimate,Tennessee!!Percent below poverty level!!Margin of Error,"Cannon County, Tennessee!!Total!!Estimate","Cannon County, Tennessee!!Total!!Margin of Error","Cannon County, Tennessee!!Below poverty level!!Estimate",...,"Warren County, Tennessee!!Below poverty level!!Estimate","Warren County, Tennessee!!Below poverty level!!Margin of Error","Warren County, Tennessee!!Percent below poverty level!!Estimate","Warren County, Tennessee!!Percent below poverty level!!Margin of Error","White County, Tennessee!!Total!!Estimate","White County, Tennessee!!Total!!Margin of Error","White County, Tennessee!!Below poverty level!!Estimate","White County, Tennessee!!Below poverty level!!Margin of Error","White County, Tennessee!!Percent below poverty level!!Estimate","White County, Tennessee!!Percent below poverty level!!Margin of Error"
0,Population for whom poverty status is determined,"6,907,625","±2,128","950,340","±15,734",13.8%,±0.2,"14,689",±23,"2,684",...,"6,532","±1,023",15.7%,±2.5,"27,812",±89,"3,664",±775,13.2%,±2.8
1,AGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Under 18 years,"1,538,193","±2,201","289,295","±8,186",18.8%,±0.5,"3,358",±111,933,...,"1,976",±514,20.4%,±5.3,"6,027",±89,"1,018",±403,16.9%,±6.7
3,Under 5 years,"402,374","±1,335","78,740","±3,029",19.6%,±0.7,843,±76,282,...,368,±179,15.0%,±7.4,"1,483",±99,379,±222,25.6%,±15.6
4,5 to 17 years,"1,135,819","±1,841","210,555","±6,349",18.5%,±0.6,"2,515",±142,651,...,"1,608",±407,22.2%,±5.6,"4,544",±126,639,±264,14.1%,±5.8


***CLEANING***

In [23]:
thda_raw['county'] = thda_raw['Geography'].str.replace(' County, Tennessee', '')

In [24]:
thda_ucdd = thda_raw.query('county in @ucdd_counties')
thda_ucdd.shape

(14, 221)

In [25]:
thda_cols = ['county', 'est_tot_pop', 'p_tot_seniors', 'p_tot_adolescent', 'p_tot_y_adult', 'p_tot_mid_age',
             'est_tot_r', 'p_r_cb', 'p_o_cb', 'p_tot_oc_cb',
             'p_tot_est_r_30p_less_ami_cb', 'p_tot_est_r_30p_50p_ami_cb', 'p_tot_est_r_50p_80p_ami_cb']

thda_clean = thda_ucdd[thda_cols]
thda_clean

,county,est_tot_pop,p_tot_seniors,p_tot_adolescent,p_tot_y_adult,p_tot_mid_age,est_tot_r,p_r_cb,p_o_cb,p_tot_oc_cb,p_tot_est_r_30p_less_ami_cb,p_tot_est_r_30p_50p_ami_cb,p_tot_est_r_50p_80p_ami_cb
7,Cannon,14818,17.714941,22.756107,18.558510,39.364287,1249,31.385108,16.920698,20.048476,53.272727,19.523810,11.320755
13,Clay,7670,24.954368,18.813559,11.368970,42.646675,893,27.323628,19.514076,21.514630,35.714286,56.470588,0.000000
17,Cumberland,63553,32.418611,16.713609,13.834123,35.567164,5581,28.507436,17.524542,19.698528,43.939394,63.582090,38.237885
20,DeKalb,20959,18.765208,21.742450,18.455079,39.090605,2370,53.670886,19.022604,28.513638,80.533333,74.234234,47.272727
24,Fentress,19309,23.294837,20.379098,15.821638,38.577865,1365,30.915751,18.620476,20.763632,56.000000,65.090909,25.070423
43,Jackson,12029,23.293707,18.289135,16.393715,39.928506,907,30.209482,16.692387,19.249218,50.769231,45.000000,9.523810
55,Macon,26240,15.392530,25.087652,18.974848,38.753811,2483,37.011679,16.964543,22.248169,70.769231,88.285714,25.606061
66,Overton,23065,20.528940,20.628658,17.871233,38.786039,2093,33.301481,16.824305,20.626171,61.111111,34.347826,29.069767
68,Pickett,5079,27.977948,17.286867,12.482772,40.756054,359,13.370474,22.068584,20.627596,38.750000,17.777778,70.000000
70,Putnam,82558,16.501126,20.862908,23.705758,34.536932,13039,45.218192,17.230814,28.091426,79.509632,77.263969,36.204934


In [26]:
thda_clean.isnull().sum()

county                         0
est_tot_pop                    0
p_tot_seniors                  0
p_tot_adolescent               0
p_tot_y_adult                  0
p_tot_mid_age                  0
est_tot_r                      0
p_r_cb                         0
p_o_cb                         0
p_tot_oc_cb                    0
p_tot_est_r_30p_less_ami_cb    0
p_tot_est_r_30p_50p_ami_cb     0
p_tot_est_r_50p_80p_ami_cb     0
dtype: int64

In [27]:
picture_raw['county'] = picture_raw['Name'].str.replace(' County', '')

In [28]:
picture_ucdd = picture_raw.query('county in @ucdd_counties').copy()
picture_ucdd.shape

(13, 85)

In [29]:
sentinel_cols = ['Subsidized units available', '# Occupied Units', '% Occupied']

for col in sentinel_cols:
    picture_ucdd.loc[picture_ucdd[col] < 0, col] = None

picture_ucdd[['county', 'Program label'] + sentinel_cols]

,county,Program label,Subsidized units available,# Occupied Units,% Occupied
0,Cannon,202/PRAC,18.0,17.0,97.0
1,Cumberland,202/PRAC,30.0,29.0,95.0
2,Cumberland,811/PRAC,17.0,12.0,70.0
3,DeKalb,202/PRAC,26.0,25.0,95.0
4,DeKalb,811/PRAC,6.0,NaN,NaN
5,Fentress,811/PRAC,20.0,19.0,97.0
6,Macon,202/PRAC,19.0,19.0,98.0
7,Macon,811/PRAC,14.0,14.0,99.0
8,Overton,202/PRAC,40.0,39.0,97.0
9,Putnam,202/PRAC,49.0,47.0,97.0


In [30]:
sec202 = picture_ucdd.query('`Program label` == "202/PRAC"')[['county', 'Subsidized units available']] \
    .rename(columns={'Subsidized units available': 'units_202'})

sec811 = picture_ucdd.query('`Program label` == "811/PRAC"')[['county', 'Subsidized units available']] \
    .rename(columns={'Subsidized units available': 'units_811'})

sec202

,county,units_202
0,Cannon,18.0
1,Cumberland,30.0
3,DeKalb,26.0
6,Macon,19.0
8,Overton,40.0
9,Putnam,49.0
10,Smith,14.0
11,Warren,14.0
12,White,55.0


In [31]:
counties_df = pd.DataFrame({'county': ucdd_counties})

picture_clean = pd.merge(counties_df, sec202, on='county', how='left')
picture_clean = pd.merge(picture_clean, sec811, on='county', how='left')

picture_clean.loc[picture_clean['units_202'].isnull(), 'units_202'] = 0
picture_clean.loc[picture_clean['units_811'].isnull(), 'units_811'] = 0
picture_clean['total_202_811_units'] = picture_clean['units_202'] + picture_clean['units_811']

picture_clean.sort_values(by='total_202_811_units', ascending=False)

,county,units_202,units_811,total_202_811_units
13,White,55.0,0.0,55.0
9,Putnam,49.0,0.0,49.0
2,Cumberland,30.0,17.0,47.0
7,Overton,40.0,0.0,40.0
6,Macon,19.0,14.0,33.0
3,DeKalb,26.0,6.0,32.0
4,Fentress,0.0,20.0,20.0
0,Cannon,18.0,0.0,18.0
10,Smith,14.0,0.0,14.0
12,Warren,14.0,0.0,14.0


In [32]:
gis_tn = gis_raw.query('STD_ST == "TN"').copy()
gis_tn.shape

(608, 8)

In [33]:
gis_tn['county'] = gis_tn['CURCNTY_NM'].str.strip()

In [34]:
gis_ucdd = gis_tn.query('county in @ucdd_counties and IS_202_811_IND == "Y"').copy()
gis_ucdd[['county', 'PROPERTY_NAME_TEXT', 'TOTAL_ASSISTED_UNIT_COUNT', 'LAT', 'LON']]

,county,PROPERTY_NAME_TEXT,TOTAL_ASSISTED_UNIT_COUNT,LAT,LON
14690,White,OAK HILLS APARTMENTS,23,35.927919,-85.452623
15460,Overton,HOLLY HILLS APARTMENTS,24,36.390450,-85.328613
16206,Fentress,Fairground Apartments,10,36.440845,-84.942345
16447,Putnam,HIGHLAND MANOR APTS.,22,36.143939,-85.260805
17286,Cumberland,MICKI THOMPSON MEMORIAL APARTMENTS,5,35.944296,-85.019317
18262,Cumberland,Oakmont Gardens Apartments,12,35.946727,-85.035685
18677,Fentress,Mace Apartments,10,36.434939,-84.944766
18716,Putnam,Laurel Creek,19,36.184155,-85.542037
19462,White,PLEASANT HILL APARTMENTS,14,35.927508,-85.451821
19466,Macon,Shenandoah Apartments,19,36.533600,-86.011800


In [35]:
gis_by_county = gis_ucdd.groupby('county')['TOTAL_ASSISTED_UNIT_COUNT'].sum().reset_index() \
    .rename(columns={'TOTAL_ASSISTED_UNIT_COUNT': 'gis_assisted_units'})

gis_by_county

,county,gis_assisted_units
0,Cannon,18
1,Cumberland,42
2,Fentress,20
3,Macon,33
4,Overton,39
5,Putnam,52
6,Smith,14
7,Warren,14
8,White,56


In [36]:
gis_clean = pd.merge(counties_df, gis_by_county, on='county', how='left')
gis_clean.loc[gis_clean['gis_assisted_units'].isnull(), 'gis_assisted_units'] = 0
gis_clean.sort_values(by='gis_assisted_units', ascending=False)

,county,gis_assisted_units
13,White,56.0
9,Putnam,52.0
2,Cumberland,42.0
7,Overton,39.0
6,Macon,33.0
4,Fentress,20.0
0,Cannon,18.0
10,Smith,14.0
12,Warren,14.0
1,Clay,0.0


In [37]:
compare = pd.merge(picture_clean[['county', 'total_202_811_units']],
                    gis_clean[['county', 'gis_assisted_units']], on='county')
compare

,county,total_202_811_units,gis_assisted_units
0,Cannon,18.0,18.0
1,Clay,0.0,0.0
2,Cumberland,47.0,42.0
3,DeKalb,32.0,0.0
4,Fentress,20.0,20.0
5,Jackson,0.0,0.0
6,Macon,33.0,33.0
7,Overton,40.0,39.0
8,Pickett,0.0,0.0
9,Putnam,49.0,52.0


In [38]:
age_sex_raw['label_clean'] = age_sex_raw['Label (Grouping)'].str.strip()
age_sex_raw[['label_clean']]

,label_clean
0,Total population
1,AGE
2,Under 5 years
3,5 to 9 years
4,10 to 14 years
5,15 to 19 years
6,20 to 24 years
7,25 to 29 years
8,30 to 34 years
9,35 to 39 years


In [39]:
total_pop_row = age_sex_raw.query('label_clean == "Total population"')
total_pop_row

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!Percent!!Estimate,Tennessee!!Percent!!Margin of Error,Tennessee!!Male!!Estimate,Tennessee!!Male!!Margin of Error,Tennessee!!Percent Male!!Estimate,Tennessee!!Percent Male!!Margin of Error,Tennessee!!Female!!Estimate,...,"White County, Tennessee!!Percent!!Margin of Error","White County, Tennessee!!Male!!Estimate","White County, Tennessee!!Male!!Margin of Error","White County, Tennessee!!Percent Male!!Estimate","White County, Tennessee!!Percent Male!!Margin of Error","White County, Tennessee!!Female!!Estimate","White County, Tennessee!!Female!!Margin of Error","White County, Tennessee!!Percent Female!!Estimate","White County, Tennessee!!Percent Female!!Margin of Error",label_clean
0,Total population,"7,066,383",*****,(X),(X),"3,467,734","±1,520",(X),(X),"3,598,649",...,(X),"13,952",±186,(X),(X),"14,208",±186,(X),(X),Total population


In [40]:
seniors_row = age_sex_raw.query('label_clean == "65 years and over"')
seniors_row

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!Percent!!Estimate,Tennessee!!Percent!!Margin of Error,Tennessee!!Male!!Estimate,Tennessee!!Male!!Margin of Error,Tennessee!!Percent Male!!Estimate,Tennessee!!Percent Male!!Margin of Error,Tennessee!!Female!!Estimate,...,"White County, Tennessee!!Percent!!Margin of Error","White County, Tennessee!!Male!!Estimate","White County, Tennessee!!Male!!Margin of Error","White County, Tennessee!!Percent Male!!Estimate","White County, Tennessee!!Percent Male!!Margin of Error","White County, Tennessee!!Female!!Estimate","White County, Tennessee!!Female!!Margin of Error","White County, Tennessee!!Percent Female!!Estimate","White County, Tennessee!!Percent Female!!Margin of Error",label_clean
31,65 years and over,"1,205,947","±1,102",17.1%,±0.1,"540,687",±700,15.6%,±0.1,"665,260",...,±0.3,"2,791",±99,20.0%,±0.6,"3,188",±102,22.4%,±0.6,65 years and over


In [41]:
total_pop_dict = {}

for county in ucdd_counties:
    col_name = county + ' County, Tennessee!!Total!!Estimate'
    value = total_pop_row[col_name].values[0]
    total_pop_dict[county] = int(value.replace(',', ''))

total_pop_dict

{'Cannon': 14818,
 'Clay': 7670,
 'Cumberland': 63553,
 'DeKalb': 20959,
 'Fentress': 19309,
 'Jackson': 12029,
 'Macon': 26240,
 'Overton': 23065,
 'Pickett': 5079,
 'Putnam': 82558,
 'Smith': 20389,
 'Van Buren': 6437,
 'Warren': 42166,
 'White': 28160}

In [42]:
seniors_pct_dict = {}

for county in ucdd_counties:
    col_name = county + ' County, Tennessee!!Percent!!Estimate'
    value = seniors_row[col_name].values[0]
    seniors_pct_dict[county] = float(value.replace('%', ''))

seniors_pct_dict

{'Cannon': 17.7,
 'Clay': 25.0,
 'Cumberland': 32.4,
 'DeKalb': 18.8,
 'Fentress': 23.3,
 'Jackson': 23.3,
 'Macon': 15.4,
 'Overton': 20.5,
 'Pickett': 28.0,
 'Putnam': 16.5,
 'Smith': 16.7,
 'Van Buren': 24.0,
 'Warren': 17.9,
 'White': 21.2}

In [43]:
age_sex_clean = pd.DataFrame(list(total_pop_dict.items()), columns=['county', 'total_population'])

pct_65_list = []
for county in age_sex_clean['county']:
    pct_65_list.append(seniors_pct_dict[county])

age_sex_clean['pct_65_and_over'] = pct_65_list
age_sex_clean

,county,total_population,pct_65_and_over
0,Cannon,14818,17.7
1,Clay,7670,25.0
2,Cumberland,63553,32.4
3,DeKalb,20959,18.8
4,Fentress,19309,23.3
5,Jackson,12029,23.3
6,Macon,26240,15.4
7,Overton,23065,20.5
8,Pickett,5079,28.0
9,Putnam,82558,16.5


In [44]:
disability_raw['label_clean'] = disability_raw['Label (Grouping)'].str.strip()
disability_raw[['label_clean']]

,label_clean
0,Total civilian noninstitutionalized population
1,SEX
2,Male
3,Female
4,RACE AND HISPANIC OR LATINO ORIGIN
...,...
68,Population 18 to 34 years
69,Population 35 to 64 years
70,Population 65 years and over
71,Population 65 to 74 years


In [45]:
disability_row = disability_raw.query('label_clean == "Total civilian noninstitutionalized population"')
disability_row

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!With a disability!!Estimate,Tennessee!!With a disability!!Margin of Error,Tennessee!!Percent with a disability!!Estimate,Tennessee!!Percent with a disability!!Margin of Error,"Cannon County, Tennessee!!Total!!Estimate","Cannon County, Tennessee!!Total!!Margin of Error","Cannon County, Tennessee!!With a disability!!Estimate",...,"Warren County, Tennessee!!With a disability!!Margin of Error","Warren County, Tennessee!!Percent with a disability!!Estimate","Warren County, Tennessee!!Percent with a disability!!Margin of Error","White County, Tennessee!!Total!!Estimate","White County, Tennessee!!Total!!Margin of Error","White County, Tennessee!!With a disability!!Estimate","White County, Tennessee!!With a disability!!Margin of Error","White County, Tennessee!!Percent with a disability!!Estimate","White County, Tennessee!!Percent with a disability!!Margin of Error",label_clean
0,Total civilian noninstitutionalized population,"6,970,655","±1,325","1,041,249","±8,879",14.9%,±0.1,"14,692",±19,"2,751",...,±601,18.0%,±1.4,"27,842",±54,"5,066",±557,18.2%,±2.0,Total civilian noninstitutionalized population


In [46]:
disability_pct_dict = {}

for county in ucdd_counties:
    col_name = county + ' County, Tennessee!!Percent with a disability!!Estimate'
    value = disability_row[col_name].values[0]
    disability_pct_dict[county] = float(value.replace('%', ''))

disability_pct_dict

{'Cannon': 18.7,
 'Clay': 23.0,
 'Cumberland': 19.4,
 'DeKalb': 19.5,
 'Fentress': 23.5,
 'Jackson': 23.9,
 'Macon': 14.7,
 'Overton': 19.0,
 'Pickett': 29.7,
 'Putnam': 13.5,
 'Smith': 15.7,
 'Van Buren': 20.7,
 'Warren': 18.0,
 'White': 18.2}

In [47]:
disability_clean = pd.DataFrame(list(disability_pct_dict.items()), columns=['county', 'pct_disability'])
disability_clean

,county,pct_disability
0,Cannon,18.7
1,Clay,23.0
2,Cumberland,19.4
3,DeKalb,19.5
4,Fentress,23.5
5,Jackson,23.9
6,Macon,14.7
7,Overton,19.0
8,Pickett,29.7
9,Putnam,13.5


In [48]:
poverty_raw['label_clean'] = poverty_raw['Label (Grouping)'].str.strip()
poverty_raw[['label_clean']]

,label_clean
0,Population for whom poverty status is determined
1,AGE
2,Under 18 years
3,Under 5 years
4,5 to 17 years
...,...
64,Mean income deficit for unrelated individuals ...
65,"Worked full-time, year-round in the past 12 mo..."
66,"Worked less than full-time, year-round in the ..."
67,Did not work


In [49]:
poverty_row = poverty_raw.query('label_clean == "Population for whom poverty status is determined"')
poverty_row

,Label (Grouping),Tennessee!!Total!!Estimate,Tennessee!!Total!!Margin of Error,Tennessee!!Below poverty level!!Estimate,Tennessee!!Below poverty level!!Margin of Error,Tennessee!!Percent below poverty level!!Estimate,Tennessee!!Percent below poverty level!!Margin of Error,"Cannon County, Tennessee!!Total!!Estimate","Cannon County, Tennessee!!Total!!Margin of Error","Cannon County, Tennessee!!Below poverty level!!Estimate",...,"Warren County, Tennessee!!Below poverty level!!Margin of Error","Warren County, Tennessee!!Percent below poverty level!!Estimate","Warren County, Tennessee!!Percent below poverty level!!Margin of Error","White County, Tennessee!!Total!!Estimate","White County, Tennessee!!Total!!Margin of Error","White County, Tennessee!!Below poverty level!!Estimate","White County, Tennessee!!Below poverty level!!Margin of Error","White County, Tennessee!!Percent below poverty level!!Estimate","White County, Tennessee!!Percent below poverty level!!Margin of Error",label_clean
0,Population for whom poverty status is determined,"6,907,625","±2,128","950,340","±15,734",13.8%,±0.2,"14,689",±23,"2,684",...,"±1,023",15.7%,±2.5,"27,812",±89,"3,664",±775,13.2%,±2.8,Population for whom poverty status is determined


In [50]:
poverty_pct_dict = {}

for county in ucdd_counties:
    col_name = county + ' County, Tennessee!!Percent below poverty level!!Estimate'
    value = poverty_row[col_name].values[0]
    poverty_pct_dict[county] = float(value.replace('%', ''))

poverty_pct_dict

{'Cannon': 18.3,
 'Clay': 24.2,
 'Cumberland': 14.4,
 'DeKalb': 18.6,
 'Fentress': 21.0,
 'Jackson': 20.3,
 'Macon': 17.2,
 'Overton': 16.3,
 'Pickett': 23.9,
 'Putnam': 18.4,
 'Smith': 10.0,
 'Van Buren': 16.1,
 'Warren': 15.7,
 'White': 13.2}

In [51]:
poverty_clean = pd.DataFrame(list(poverty_pct_dict.items()), columns=['county', 'pct_poverty'])
poverty_clean

,county,pct_poverty
0,Cannon,18.3
1,Clay,24.2
2,Cumberland,14.4
3,DeKalb,18.6
4,Fentress,21.0
5,Jackson,20.3
6,Macon,17.2
7,Overton,16.3
8,Pickett,23.9
9,Putnam,18.4


In [52]:
master = counties_df.copy()
master = pd.merge(master, thda_clean, on='county', how='left')
master = pd.merge(master, picture_clean, on='county', how='left')
master = pd.merge(master, gis_clean, on='county', how='left')
master = pd.merge(master, age_sex_clean, on='county', how='left')
master = pd.merge(master, disability_clean, on='county', how='left')
master = pd.merge(master, poverty_clean, on='county', how='left')

master

,county,est_tot_pop,p_tot_seniors,p_tot_adolescent,p_tot_y_adult,p_tot_mid_age,est_tot_r,p_r_cb,p_o_cb,p_tot_oc_cb,...,p_tot_est_r_30p_50p_ami_cb,p_tot_est_r_50p_80p_ami_cb,units_202,units_811,total_202_811_units,gis_assisted_units,total_population,pct_65_and_over,pct_disability,pct_poverty
0,Cannon,14818,17.714941,22.756107,18.558510,39.364287,1249,31.385108,16.920698,20.048476,...,19.523810,11.320755,18.0,0.0,18.0,18.0,14818,17.7,18.7,18.3
1,Clay,7670,24.954368,18.813559,11.368970,42.646675,893,27.323628,19.514076,21.514630,...,56.470588,0.000000,0.0,0.0,0.0,0.0,7670,25.0,23.0,24.2
2,Cumberland,63553,32.418611,16.713609,13.834123,35.567164,5581,28.507436,17.524542,19.698528,...,63.582090,38.237885,30.0,17.0,47.0,42.0,63553,32.4,19.4,14.4
3,DeKalb,20959,18.765208,21.742450,18.455079,39.090605,2370,53.670886,19.022604,28.513638,...,74.234234,47.272727,26.0,6.0,32.0,0.0,20959,18.8,19.5,18.6
4,Fentress,19309,23.294837,20.379098,15.821638,38.577865,1365,30.915751,18.620476,20.763632,...,65.090909,25.070423,0.0,20.0,20.0,20.0,19309,23.3,23.5,21.0
5,Jackson,12029,23.293707,18.289135,16.393715,39.928506,907,30.209482,16.692387,19.249218,...,45.000000,9.523810,0.0,0.0,0.0,0.0,12029,23.3,23.9,20.3
6,Macon,26240,15.392530,25.087652,18.974848,38.753811,2483,37.011679,16.964543,22.248169,...,88.285714,25.606061,19.0,14.0,33.0,33.0,26240,15.4,14.7,17.2
7,Overton,23065,20.528940,20.628658,17.871233,38.786039,2093,33.301481,16.824305,20.626171,...,34.347826,29.069767,40.0,0.0,40.0,39.0,23065,20.5,19.0,16.3
8,Pickett,5079,27.977948,17.286867,12.482772,40.756054,359,13.370474,22.068584,20.627596,...,17.777778,70.000000,0.0,0.0,0.0,0.0,5079,28.0,29.7,23.9
9,Putnam,82558,16.501126,20.862908,23.705758,34.536932,13039,45.218192,17.230814,28.091426,...,77.263969,36.204934,49.0,0.0,49.0,52.0,82558,16.5,13.5,18.4


In [53]:
master.isnull().sum()

county                         0
est_tot_pop                    0
p_tot_seniors                  0
p_tot_adolescent               0
p_tot_y_adult                  0
p_tot_mid_age                  0
est_tot_r                      0
p_r_cb                         0
p_o_cb                         0
p_tot_oc_cb                    0
p_tot_est_r_30p_less_ami_cb    0
p_tot_est_r_30p_50p_ami_cb     0
p_tot_est_r_50p_80p_ami_cb     0
units_202                      0
units_811                      0
total_202_811_units            0
gis_assisted_units             0
total_population               0
pct_65_and_over                0
pct_disability                 0
pct_poverty                    0
dtype: int64

In [54]:
master.describe()

,est_tot_pop,p_tot_seniors,p_tot_adolescent,p_tot_y_adult,p_tot_mid_age,est_tot_r,p_r_cb,p_o_cb,p_tot_oc_cb,p_tot_est_r_30p_less_ami_cb,p_tot_est_r_30p_50p_ami_cb,p_tot_est_r_50p_80p_ami_cb,units_202,units_811,total_202_811_units,gis_assisted_units,total_population,pct_65_and_over,pct_disability,pct_poverty
count,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000
mean,26602.285714,21.475517,20.723479,16.922907,38.830797,2827.214286,32.054156,17.532309,21.299659,56.257115,57.645490,28.612348,18.928571,4.071429,23.000000,20.571429,26602.285714,21.478571,19.821429,17.685714
std,22286.443342,4.860525,2.357655,3.100553,2.035888,3290.401219,9.569872,1.820790,3.253344,16.562587,21.630344,18.086413,18.939029,7.279733,19.725033,20.391228,22286.443342,4.860069,4.205889,3.914442
min,5079.000000,15.392530,16.713609,11.368970,34.536932,349.000000,13.370474,15.155904,16.660179,26.222222,17.777778,0.000000,0.000000,0.000000,0.000000,0.000000,5079.000000,15.400000,13.500000,10.000000
25%,12726.250000,17.763747,19.170757,15.408726,38.329307,992.500000,28.101435,16.725366,19.786015,45.646853,47.812500,16.782297,0.000000,0.000000,3.500000,0.000000,12726.250000,17.750000,18.050000,15.800000
50%,20674.000000,20.880592,20.745783,17.657208,38.769925,1962.500000,30.562616,17.097679,20.626884,56.366013,61.033469,27.055904,16.000000,0.000000,19.000000,16.000000,20674.000000,20.850000,19.200000,17.750000
75%,27680.000000,23.825107,22.231434,18.514187,39.787451,2567.750000,36.084130,18.394674,21.437460,68.354701,73.819136,37.729647,29.000000,4.500000,38.250000,37.500000,27680.000000,23.825000,22.425000,19.875000
max,82558.000000,32.418611,25.087652,23.705758,42.646675,13039.000000,53.670886,22.068584,28.513638,80.533333,88.285714,70.000000,55.000000,20.000000,55.000000,56.000000,82558.000000,32.400000,29.700000,24.200000


In [55]:
master.to_csv('ucdd_master_county_data.csv', index=False)

In [56]:
gis_ucdd.to_csv('ucdd_property_level_202_811.csv', index=False)